In [1]:
import pandas as pd
import requests
import time
import warnings
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS request.*")

# -----------------------------------
# INPUT FILES
# -----------------------------------
INPUT_FILES = [
    "final_merged_5_geocoded_formatted_address_enriched.csv",
    "final_merged_24_geocoded_formatted_address_enriched.csv",
]

# -----------------------------------
# ACS CONFIG
# -----------------------------------
# user wants:
#   2015 -> ACS 2011-2015 5-year -> API year 2015
#   2020 -> ACS 2016-2020 5-year -> API year 2020
#   2025 -> use latest available proxy -> ACS 2020-2024 5-year -> API year 2024
ACS_YEARS = {
    "2015": 2015,
    "2020": 2020,
    "2025": 2024,   # proxy for 2025
}

# Variables:
# B19013_001E = median household income
# B03002_* = race / ethnicity counts
# C17002_001E total, _002E under .50, _003E .50 to .99
ACS_VARS = [
    "B19013_001E",
    "B03002_001E",  # total
    "B03002_003E",  # non-Hispanic White alone
    "B03002_004E",  # non-Hispanic Black alone
    "B03002_006E",  # non-Hispanic Asian alone
    "B03002_012E",  # Hispanic or Latino
    "C17002_001E",  # total for poverty denominator
    "C17002_002E",  # under .50
    "C17002_003E",  # .50 to .99
]

# -----------------------------------
# HELPERS
# -----------------------------------
def tract_code_6_from_row(row):
    """
    Prefer full 11-digit GEOID from 'Census Tract' if available.
    Fallback to 'census_tract_final' / 'census_tract_geocoder'.

    DC tract GEOID format:
      state(2)=11 + county(3)=001 + tract(6)
    so tract-only code is last 6 digits.
    """
    # 1. full GEOID from Census Tract
    if "Census Tract" in row.index and pd.notna(row["Census Tract"]):
        value = str(row["Census Tract"]).strip()
        value = value.replace(".0", "")
        digits = "".join(ch for ch in value if ch.isdigit())

        if len(digits) >= 11:
            return digits[-6:]

    # 2. fallback to census_tract_final
    for col in ["census_tract_final", "census_tract_geocoder"]:
        if col in row.index and pd.notna(row[col]):
            value = str(row[col]).strip()
            value = value.replace(".0", "")
            digits = "".join(ch for ch in value if ch.isdigit())

            if len(digits) <= 6 and len(digits) > 0:
                return digits.zfill(6)
            if len(digits) > 6:
                return digits[-6:]

    return None


def safe_num(x):
    try:
        return float(x)
    except:
        return None


def fetch_dc_tract_data(api_year):
    """
    Pull all DC census tracts from ACS 5-year API for one year.
    """
    url = f"https://api.census.gov/data/{api_year}/acs/acs5"
    params = {
        "get": ",".join(ACS_VARS),
        "for": "tract:*",
        "in": "state:11 county:001",  # DC
    }

    r = requests.get(url, params=params, timeout=120, verify=False)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data[1:], columns=data[0])

    # numeric conversions
    for col in ACS_VARS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # tract key
    df["tract_code_6"] = df["tract"].astype(str).str.zfill(6)
    df["tract_geoid"] = df["state"].astype(str) + df["county"].astype(str) + df["tract_code_6"]

    # rename core columns
    suffix = str(api_year)
    df = df.rename(columns={
        "B19013_001E": f"median_income_api_{suffix}",
        "B03002_001E": f"race_total_api_{suffix}",
        "B03002_003E": f"nh_white_api_{suffix}",
        "B03002_004E": f"nh_black_api_{suffix}",
        "B03002_006E": f"nh_asian_api_{suffix}",
        "B03002_012E": f"hispanic_api_{suffix}",
        "C17002_001E": f"poverty_total_api_{suffix}",
        "C17002_002E": f"poverty_under_050_api_{suffix}",
        "C17002_003E": f"poverty_050_099_api_{suffix}",
    })

    # poverty rate
    df[f"poverty_count_api_{suffix}"] = (
        df[f"poverty_under_050_api_{suffix}"] + df[f"poverty_050_099_api_{suffix}"]
    )
    df[f"poverty_rate_api_{suffix}"] = (
        df[f"poverty_count_api_{suffix}"] / df[f"poverty_total_api_{suffix}"]
    )

    # race shares
    for group in ["nh_white", "nh_black", "nh_asian", "hispanic"]:
        df[f"{group}_share_api_{suffix}"] = (
            df[f"{group}_api_{suffix}"] / df[f"race_total_api_{suffix}"]
        )

    keep_cols = [
        "tract_code_6",
        "tract_geoid",
        f"median_income_api_{suffix}",
        f"race_total_api_{suffix}",
        f"nh_white_api_{suffix}",
        f"nh_black_api_{suffix}",
        f"nh_asian_api_{suffix}",
        f"hispanic_api_{suffix}",
        f"nh_white_share_api_{suffix}",
        f"nh_black_share_api_{suffix}",
        f"nh_asian_share_api_{suffix}",
        f"hispanic_share_api_{suffix}",
        f"poverty_total_api_{suffix}",
        f"poverty_count_api_{suffix}",
        f"poverty_rate_api_{suffix}",
    ]
    return df[keep_cols]


def add_pretty_year_columns(df):
    """
    Rename api-year columns to user-facing year labels:
      2015 -> 2015
      2020 -> 2020
      2024 proxy -> 2025
    """
    rename_map = {
        "median_income_api_2015": "median_income_2015",
        "race_total_api_2015": "race_total_2015",
        "nh_white_api_2015": "nh_white_2015",
        "nh_black_api_2015": "nh_black_2015",
        "nh_asian_api_2015": "nh_asian_2015",
        "hispanic_api_2015": "hispanic_2015",
        "nh_white_share_api_2015": "nh_white_share_2015",
        "nh_black_share_api_2015": "nh_black_share_2015",
        "nh_asian_share_api_2015": "nh_asian_share_2015",
        "hispanic_share_api_2015": "hispanic_share_2015",
        "poverty_total_api_2015": "poverty_total_2015",
        "poverty_count_api_2015": "poverty_count_2015",
        "poverty_rate_api_2015": "poverty_rate_2015",

        "median_income_api_2020": "median_income_2020",
        "race_total_api_2020": "race_total_2020",
        "nh_white_api_2020": "nh_white_2020",
        "nh_black_api_2020": "nh_black_2020",
        "nh_asian_api_2020": "nh_asian_2020",
        "hispanic_api_2020": "hispanic_2020",
        "nh_white_share_api_2020": "nh_white_share_2020",
        "nh_black_share_api_2020": "nh_black_share_2020",
        "nh_asian_share_api_2020": "nh_asian_share_2020",
        "hispanic_share_api_2020": "hispanic_share_2020",
        "poverty_total_api_2020": "poverty_total_2020",
        "poverty_count_api_2020": "poverty_count_2020",
        "poverty_rate_api_2020": "poverty_rate_2020",

        # 2025 requested -> use 2024 ACS proxy
        "median_income_api_2024": "median_income_2025_proxy_2024acs",
        "race_total_api_2024": "race_total_2025_proxy_2024acs",
        "nh_white_api_2024": "nh_white_2025_proxy_2024acs",
        "nh_black_api_2024": "nh_black_2025_proxy_2024acs",
        "nh_asian_api_2024": "nh_asian_2025_proxy_2024acs",
        "hispanic_api_2024": "hispanic_2025_proxy_2024acs",
        "nh_white_share_api_2024": "nh_white_share_2025_proxy_2024acs",
        "nh_black_share_api_2024": "nh_black_share_2025_proxy_2024acs",
        "nh_asian_share_api_2024": "nh_asian_share_2025_proxy_2024acs",
        "hispanic_share_api_2024": "hispanic_share_2025_proxy_2024acs",
        "poverty_total_api_2024": "poverty_total_2025_proxy_2024acs",
        "poverty_count_api_2024": "poverty_count_2025_proxy_2024acs",
        "poverty_rate_api_2024": "poverty_rate_2025_proxy_2024acs",
    }
    return df.rename(columns=rename_map)


# -----------------------------------
# FETCH ACS LOOKUPS ONCE
# -----------------------------------
lookup_frames = []
for label, api_year in ACS_YEARS.items():
    print(f"Downloading ACS tract data for requested year {label} using API year {api_year}...")
    lookup_frames.append(fetch_dc_tract_data(api_year))
    time.sleep(0.5)

acs_lookup = lookup_frames[0]
for extra in lookup_frames[1:]:
    acs_lookup = acs_lookup.merge(extra, on=["tract_code_6", "tract_geoid"], how="outer")

acs_lookup = add_pretty_year_columns(acs_lookup)

# -----------------------------------
# APPLY TO EACH FILE
# -----------------------------------
for file_path in INPUT_FILES:
    print(f"\nProcessing {file_path} ...")
    df = pd.read_csv(file_path, low_memory=False)

    # build tract join keys
    df["tract_code_6"] = df.apply(tract_code_6_from_row, axis=1)
    df["tract_geoid"] = df["tract_code_6"].apply(
        lambda x: f"11001{x}" if pd.notna(x) else None
    )

    # merge tract-level ACS data
    df = df.merge(
        acs_lookup,
        on=["tract_code_6", "tract_geoid"],
        how="left"
    )

    # optional percentage formatting copies
    share_cols = [
        "nh_white_share_2015", "nh_black_share_2015", "nh_asian_share_2015", "hispanic_share_2015", "poverty_rate_2015",
        "nh_white_share_2020", "nh_black_share_2020", "nh_asian_share_2020", "hispanic_share_2020", "poverty_rate_2020",
        "nh_white_share_2025_proxy_2024acs", "nh_black_share_2025_proxy_2024acs",
        "nh_asian_share_2025_proxy_2024acs", "hispanic_share_2025_proxy_2024acs",
        "poverty_rate_2025_proxy_2024acs",
    ]
    for col in share_cols:
        if col in df.columns:
            df[col + "_pct"] = df[col] * 100

    output_path = file_path.replace(".csv", "_with_income_race_poverty.csv")
    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    print("Rows matched:", df["median_income_2015"].notna().sum(), "of", len(df))


Processing final_merged_5_geocoded_formatted_address_enriched.csv ...
Saved: final_merged_5_geocoded_formatted_address_enriched_with_income_race_poverty.csv
Rows matched: 712 of 893

Processing final_merged_24_geocoded_formatted_address_enriched.csv ...
Saved: final_merged_24_geocoded_formatted_address_enriched_with_income_race_poverty.csv
Rows matched: 1619 of 1821
